# Toxicity in Indian esports YouTube comments

Runs the whole study: collect comments, score them, label a sample by hand, then measure and report.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.* Scoring is about 10× faster on a GPU.
2. Have a YouTube Data API key ready (free; see the README).

Steps 1–4 take about 15 minutes. Step 5 is the hand labelling, then step 6 produces the results.

In [ ]:
# 1. Get the code (safe to re-run: updates the existing copy instead of cloning again)
import os
os.chdir("/content")
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull -q
else:
    !git clone -q https://github.com/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels.git /content/repo
%cd -q /content/repo
!pip install -q -e ".[model]"
print("Ready in", os.getcwd())


In [ ]:
# 2. API key: stored as a Colab secret named YOUTUBE_API_KEY (key icon in the left sidebar), or typed here
import os, getpass
try:
    from google.colab import userdata
    os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")
except Exception:
    os.environ["YOUTUBE_API_KEY"] = getpass.getpass("YouTube API key: ")

In [ ]:
%cd -q /content/repo
# 3. Collect comments (edit config/channels.yaml first if a handle needs fixing)
!python -m toxicity collect

In [ ]:
%cd -q /content/repo
# 4. Score every comment with the model and the word list, then draw the labelling sample
!python -m toxicity score
!python -m toxicity sample -n 400
from google.colab import files
try:
    files.download("data/annotation/to_label.csv")
except Exception:
    pass
print("If no download started: open the Files panel (folder icon, left), go to repo/data/annotation,\n"
      "and right-click to_label.csv → Download.")

## 5. Label the sample

Open `to_label.csv` in Google Sheets or Excel and fill the **toxic** column with `1` or `0` for every row,
following [docs/labelling-guide.md](https://github.com/harsh-github007/Identifying-Toxicity-Within-Esports-YouTube-Channels/blob/main/docs/labelling-guide.md).
The optional **category** column takes `insult`, `profanity`, `threat` or `hate`.

The sheet doesn't show which channel a comment came from or what the model thought, so the labels stay independent.
Save it as CSV, then upload it with the next cell.

In [ ]:
%cd -q /content/repo
# Upload your labelled file
import os
import pandas as pd
from google.colab import files
up = files.upload()
name = next(iter(up))
os.makedirs("data/annotation", exist_ok=True)
os.replace(name, "data/annotation/labels.csv")
lab = pd.read_csv("data/annotation/labels.csv", dtype=str, sep=None, engine="python", encoding="utf-8-sig")
done = lab["toxic"].fillna("").str.strip().ne("").sum()
print(f"{done} of {len(lab)} comments labelled ({(lab['toxic'].str.strip() == '1').sum()} toxic)")
if done < 50:
    print("Label more rows before running the next cell. At least 50 are needed; all 400 is best.")


In [ ]:
%cd -q /content/repo
# 6. Measure the methods, estimate toxicity per channel, draw the charts
!python -m toxicity evaluate
import os
from IPython.display import Markdown, Image, display
if os.path.exists("results/results.md"):
    display(Markdown(open("results/results.md").read().replace("figures/", "results/figures/")))
    display(Image("results/figures/prevalence.png")); display(Image("results/figures/methods.png"))
else:
    print("No results yet. Fix the message above, then run this cell again.")


In [ ]:
%cd -q /content/repo
# 7. Download the results to add to the repository (results/ and your labels)
from google.colab import files
!zip -qr results_to_commit.zip results data/annotation/labels.csv data/annotation/sample_key.csv config
files.download("results_to_commit.zip")